In [1]:
import pandas as pd
import pyxlsb
import numpy as np
import plotly.express as px

file_path = "stc TV Data Set_T1.xlsb"
dataframe = pd.read_excel(file_path, sheet_name="Final_Dataset")

df = dataframe.copy()

print("=== 1. أبعاد البيانات (الصفوف، الأعمدة) ===")
print(df.shape)

df = df.drop(columns=['Column1'])
df['program_name'] = df['program_name'].str.strip()
df['date_'] = pd.to_datetime(df['date_'], unit='D', origin='30/12/1899')

numeric_cols = ['duration_seconds', 'season', 'episode', 'series_title', 'hd']
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

string_cols = ['user_id_maped', 'program_name', 'program_class', 'program_desc', 'program_genre', 'original_name']
df[string_cols] = df[string_cols].astype(str)

print("\n=== 2. عدد القيم الفارغة في الأعمدة ===")
print(df.isnull().sum())

print("\n=== 3. الوصف الإحصائي (المتوسط، الانحراف، العظمى، والصغرى) ===")
print(df.describe())

grouped = df.copy()

grouped.loc[grouped['program_class'] == 'SERIES/EPISODES', 'program_name'] = (
    grouped['program_name'] + '_SE' + grouped['season'].astype(str) + '_EP' + grouped['episode'].astype(str)
)

grouped = grouped.groupby(['program_name', 'program_class']).agg(
    {'user_id_maped': [('No of Users who Watched', 'nunique'), ('No of watches', 'count')],
     'duration_seconds': [('Total watch time in seconds', 'sum')]}
).reset_index()

grouped.columns = ['program_name', 'program_class', 'No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped['Total watch time in houres'] = grouped['Total watch time in seconds'] / 3600
grouped = grouped.drop(columns=['Total watch time in seconds'])

top_10 = grouped.sort_values(by=['Total watch time in houres', 'No of watches', 'No of Users who Watched'], ascending=False).head(10).reset_index(drop=True)

print("\n=== 4. قائمة Top 10 للبرامج الأكثر مشاهدة ===")
print(top_10)

fig_top10 = px.pie(top_10, values='Total watch time in houres', names='program_name',
                   hover_data=['program_class'], title='أفضل 10 برامج في إجمالي ساعات المشاهدة')
fig_top10.show()

grouped_class = df.groupby('program_class').agg(
    {'user_id_maped': [('No of Users who Watched', 'nunique'), ('No of watches', 'count')],
     'duration_seconds': [('Total watch time in seconds', 'sum')]}
).reset_index()

grouped_class.columns = ['program_class', 'No of Users who Watched', 'No of watches', 'Total watch time in seconds']
grouped_class['Total watch time in houres'] = grouped_class['Total watch time in seconds'] / 3600

fig_class_time = px.pie(grouped_class, values='Total watch time in houres', names='program_class', title='إجمالي وقت المشاهدة حسب فئة المحتوى')
fig_class_users = px.pie(grouped_class, values='No of Users who Watched', names='program_class', title='إجمالي عدد المستخدمين حسب فئة المحتوى')

fig_class_time.show()
fig_class_users.show()

hd_grouped = df.groupby('hd').agg(
    {'user_id_maped': [('No of Users who Watched', 'nunique'), ('No of watches', 'count')],
     'duration_seconds': [('Total watch time in seconds', 'sum')]}
).reset_index()

hd_grouped.columns = ['HD Flag', 'No of Users who Watched', 'No of watches', 'Total watch time in seconds']
hd_grouped['Total watch time in hours'] = hd_grouped['Total watch time in seconds'] / 3600
hd_grouped['HD Label'] = hd_grouped['HD Flag'].map({1: 'HD', 0: 'SD'})

print("\n=== 5. تحليل الجودة HD vs SD ===")
print(hd_grouped[['HD Label', 'No of Users who Watched', 'No of watches', 'Total watch time in hours']])

fig_hd_time = px.pie(
    hd_grouped,
    values='Total watch time in hours',
    names='HD Label',
    title='إجمالي وقت المشاهدة حسب الجودة (HD vs SD)'
)

fig_hd_users = px.bar(
    hd_grouped,
    x='HD Label',
    y='No of Users who Watched',
    color='HD Label',
    title='عدد المستخدمين الفريدين لكل جودة (HD vs SD)',
    text_auto=True
)

fig_hd_time.show()
fig_hd_users.show()

movies = df[df['program_class'] == 'MOVIE']

total_movies = len(movies)
hd_movies = movies['hd'].sum()
sd_movies = total_movies - hd_movies

print(f"إجمالي مشاهدات الأفلام: {total_movies:,}")
print(f"مشاهدات الأفلام HD: {hd_movies:,}")
print(f"مشاهدات الأفلام SD: {sd_movies:,}")
print(f"النسبة المئوية لـ HD: {(hd_movies / total_movies) * 100:.1f}%")

=== 1. أبعاد البيانات (الصفوف، الأعمدة) ===
(1048575, 13)

=== 2. عدد القيم الفارغة في الأعمدة ===
date_                   0
user_id_maped           0
program_name            0
duration_seconds        0
program_class           0
season                  0
episode                 0
program_desc        14038
program_genre           0
series_title            0
hd                      0
original_name           0
dtype: int64

=== 3. الوصف الإحصائي (المتوسط، الانحراف، العظمى، والصغرى) ===
                     date_  duration_seconds        season       episode  \
count              1048575      1.048575e+06  1.048575e+06  1.048575e+06   
mean   2017-10-04 00:23:20      1.230957e+03  1.342139e+00  6.157952e+00   
min    2017-03-14 00:00:00      2.000000e+00  0.000000e+00  0.000000e+00   
25%    2017-06-10 00:00:00      5.200000e+01  0.000000e+00  0.000000e+00   
50%    2017-10-14 00:00:00      1.190000e+02  1.000000e+00  1.000000e+00   
75%    2018-01-21 00:00:00      1.328000e+03  1.000000e+


=== 5. تحليل الجودة HD vs SD ===
  HD Label  No of Users who Watched  No of watches  Total watch time in hours
0       SD                     6728         643539              268364.372778
1       HD                    11000         405036               90177.560278


إجمالي مشاهدات الأفلام: 488,401
مشاهدات الأفلام HD: 331,746
مشاهدات الأفلام SD: 156,655
النسبة المئوية لـ HD: 67.9%
